# Import Library

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import joblib

# Import Dataset (Train & Test)

In [ ]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

IOT_NORMAL_PATH = BASE_DIR / "iot_normal.csv"
IOT_ANOMALY_PATH = BASE_DIR / "iot_anomaly.csv"
VOLTAGE_NORMAL_PATH = BASE_DIR / "voltage_normal.csv"
VOLTAGE_ANOMALY_PATH = BASE_DIR / "voltage_anomaly.csv"

MODEL_DIR = BASE_DIR / "models"
OUTPUT_DIR = BASE_DIR / "outputs"
DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed"

MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
iot_normal = pd.read_csv(IOT_NORMAL_PATH)
iot_anomaly = pd.read_csv(IOT_ANOMALY_PATH)
voltage_normal = pd.read_csv(VOLTAGE_NORMAL_PATH)
voltage_anomaly = pd.read_csv(VOLTAGE_ANOMALY_PATH)

In [ ]:
iot_normal["label"] = 0
voltage_normal["label"] = 0

iot_anomaly["label"] = 1
voltage_anomaly["label"] = 1

dataset = pd.concat(
    [iot_normal, iot_anomaly, voltage_normal, voltage_anomaly],
    ignore_index=True
)

dataset.to_csv(DATA_PROCESSED_DIR / "dataset.csv", index=False)

dataset.head()

# Exploratory Data Analysis

In [ ]:
dataset.shape

In [ ]:
dataset.info()

In [ ]:
dataset.isna().sum()

# Processing

In [ ]:
X = dataset.drop(columns=["label"])
y = dataset["label"]

In [ ]:
X = pd.get_dummies(X, drop_first=True)

In [ ]:
X = X.fillna(0)

In [ ]:
feature_columns = X.columns.tolist()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

## Modelling

In [ ]:
# DECISION TREE

dt = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred_dt))
print(classification_report(y_val, y_pred_dt, target_names=["normal", "anomaly"]))

In [ ]:
# LOGISTIC REGRESSION

lr = LogisticRegression(max_iter=1000, random_state=42)

lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_val_scaled)

print("Accuracy:", accuracy_score(y_val, y_pred_lr))
print(classification_report(y_val, y_pred_lr, target_names=["normal", "anomaly"]))

In [ ]:
# RANDOM FOREST

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred_rf))
print(classification_report(y_val, y_pred_rf, target_names=["normal", "anomaly"]))

In [ ]:
# GRADIENT BOOSTING

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred_gb))
print(classification_report(y_val, y_pred_gb, target_names=["normal", "anomaly"]))

# **Perbandingan Model**

In [ ]:
model_scores = {
    "Decision Tree": accuracy_score(y_val, y_pred_dt),
    "Logistic Regression": accuracy_score(y_val, y_pred_lr),
    "Random Forest": accuracy_score(y_val, y_pred_rf),
    "Gradient Boosting": accuracy_score(y_val, y_pred_gb),
}

model_ranking = pd.DataFrame(
    model_scores.items(),
    columns=["Model", "Accuracy"]
).sort_values(by="Accuracy", ascending=False)

model_ranking

In [ ]:
best_model = rf
best_model_name = "Random Forest"

print("Model terbaik yang digunakan:", best_model_name)

# **Simpan Model**

In [ ]:
MODEL_PATH = MODEL_DIR / "anomaly_detection_model.pkl"
SCALER_PATH = MODEL_DIR / "scaler.pkl"
FEATURE_COLUMNS_PATH = MODEL_DIR / "feature_columns.pkl"

joblib.dump(best_model, MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)
joblib.dump(feature_columns, FEATURE_COLUMNS_PATH)

print("Model disimpan di:", MODEL_PATH)
print("Scaler disimpan di:", SCALER_PATH)
print("Feature columns disimpan di:", FEATURE_COLUMNS_PATH)

# **Template Fungsi Prediksi**

In [ ]:
def predict_from_csv(input_path, output_path=OUTPUT_DIR / "prediction_result.csv"):
    input_data = pd.read_csv(input_path)

    input_features = input_data.copy()

    if "label" in input_features.columns:
        input_features = input_features.drop(columns=["label"])

    input_features = pd.get_dummies(input_features, drop_first=True)
    input_features = input_features.fillna(0)

    input_features = input_features.reindex(columns=feature_columns, fill_value=0)

    predictions = best_model.predict(input_features)

    result = input_data.copy()
    result["prediction"] = predictions
    result["prediction_label"] = result["prediction"].map({
        0: "normal",
        1: "anomaly"
    })

    result.to_csv(output_path, index=False)

    return result

# **Ensemble Methods**

# **Prediksi Data Test**